# Crate — LoRA fine-tune of CLAP on producer vocab (Colab)

Runtime → **GPU**. Set two Colab secrets (key icon, left sidebar): `FREESOUND_KEY` and `HF_TOKEN`.
Trains the LoRA adapter and pushes it to your HF repo.

In [ ]:
!git clone https://github.com/jahnavi-yelamanchi/crate.git
%cd crate
!pip -q install -e ".[train]"

In [ ]:
import os
from google.colab import userdata
os.environ['FREESOUND_KEY'] = userdata.get('FREESOUND_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HF_MODEL_REPO'] = 'jahnavi-yelamanchi/crate-clap-lora'  # <-- your repo id

In [ ]:
# Data pipeline: pull licensing-clean audio, preprocess, build pairs.
# Skip this cell if you've uploaded a prebuilt data/ dir instead.
!bash scripts/ingest.sh

In [ ]:
from crate.model.lora import train
train(epochs=4, batch_size=16, augment_prob=0.5, push_to_hub=True)

In [ ]:
# Eval: fine-tuned vs base CLAP on held-out producer-vocab pairs, then republish card.
!python eval/recall_at_k.py
!python scripts/publish_hf.py